# 01 - S4D

**Model 1 of 14 in the lineage** (see `docs/00_research_lineage.md` for the
full map). Foundation tier: this is the model every later one in this repo
implicitly compares itself against.

### The recurrence

$$x_t = \bar{A} x_{t-1} + \bar{B} u_t \qquad y_t = \mathrm{Re}(C x_t) + D u_t$$

$\bar A, \bar B$ are the zero-order-hold discretization of a diagonal,
complex, **input-independent** state matrix $A$ (initialized from a diagonal
approximation to the HiPPO-LegS spectrum -- S4D-Lin here). Every channel
runs its own independent $N$-dimensional linear recurrence; the associative
scan in `ssm_lab.utils.scan` evaluates all of them for a full sequence in
$O(\log L)$ sequential steps instead of $L$.

**References:** Gu, Goel & Re, *Efficiently Modeling Long Sequences with
Structured State Spaces* (S4, ICLR 2022); Gu, Gupta, Goel & Re, *On the
Parameterization and Initialization of Diagonal State Space Models* (S4D,
NeurIPS 2022); Gu et al., *HiPPO: Recurrent Memory with Optimal Polynomial
Projections* (NeurIPS 2020).

**Honest scope note:** full S4 evaluates a *non-diagonal* state matrix
subquadratically via a Cauchy-kernel/Woodbury trick over an NPLR
decomposition. We implement the diagonal simplification (S4D) instead --
see the docstring in `src/ssm_lab/layers/s4d.py` and `docs/01_s4.md` for
exactly what that trades away. The one property that matters most for
everything below: $\bar A$ here is a learned *constant*, not a function of
the input. Model 3 in this repo (Mamba) changes exactly that one thing.

## Setup

Run this cell first, every time. It detects Colab vs. Kaggle vs. local,
gets `ssm_lab` importable either way, and installs dependencies if needed.
Fill in `REPO_URL` below once you've pushed this repo to your own GitHub.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/YOUR_USERNAME/state-space-models.git"  # <- set this after pushing


def _in_colab() -> bool:
    return "google.colab" in sys.modules


def _in_kaggle() -> bool:
    return "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle")


IN_COLAB, IN_KAGGLE = _in_colab(), _in_kaggle()
IN_HOSTED_NOTEBOOK = IN_COLAB or IN_KAGGLE
print(f"Environment -> Colab: {IN_COLAB} | Kaggle: {IN_KAGGLE} | Local: {not IN_HOSTED_NOTEBOOK}")


def ensure_repo_on_path() -> None:
    try:
        import ssm_lab  # noqa: F401

        print("ssm_lab already importable -- nothing to do.")
        return
    except ImportError:
        pass

    for candidate in ["state-space-models", "/kaggle/working/state-space-models", "."]:
        if os.path.isdir(os.path.join(candidate, "src", "ssm_lab")):
            sys.path.insert(0, os.path.join(candidate, "src"))
            print(f"Found repo at '{candidate}', added src/ to sys.path.")
            return

    if IN_HOSTED_NOTEBOOK:
        print("Repo not found locally -- cloning...")
        subprocess.run(["git", "clone", "--quiet", REPO_URL], check=True)
        sys.path.insert(0, "state-space-models/src")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", "state-space-models/requirements.txt"],
            check=True,
        )
    else:
        raise RuntimeError(
            "Could not find ssm_lab. Run this notebook from inside the repo "
            "(e.g. `jupyter notebook` from the repo root), or set REPO_URL "
            "above and re-run."
        )


ensure_repo_on_path()
import ssm_lab  # noqa: E402

print("ssm_lab imported OK from:", ssm_lab.__file__)


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print(
        "No GPU detected. On Kaggle: Settings -> Accelerator -> GPU T4 x2 or P100. "
        "On Colab: Runtime -> Change runtime type -> T4 GPU. "
        "CPU works fine for FAST_DEV_RUN=True."
    )


## Fast dev run

`FAST_DEV_RUN = True` shrinks every config below to a ~10-30 second CPU-friendly
sanity check (tiny model, tiny sequences, ~100 steps) -- use it to confirm the
whole notebook runs cleanly before committing GPU time to it. Set it to
`False` for the real run this notebook is meant to produce.

In [ ]:
FAST_DEV_RUN = True  # set False for the real run


## Shared run helper

Every benchmark section below calls this one function: fresh model, fresh
`Trainer`, same recipe. This -- not any per-benchmark special-casing -- is
what keeps the comparison in `20_benchmark_suite.ipynb` fair once all 14
models exist.

In [ ]:
import json
import time

import matplotlib.pyplot as plt

from ssm_lab.layers.s4d import S4DLayer
from ssm_lab.models.backbone import BackboneConfig, SequenceBackbone
from ssm_lab.training.trainer import Trainer, TrainConfig

MODEL_NAME = "s4d"
ALL_RESULTS = {}  # populated by every section below, saved to results/ at the end


def run_benchmark(
    tag, vocab_size, train_fn, val_fn,
    d_model, d_state, n_layers, max_steps, lr=3e-3, eval_iters=10, plot=True, seed=0,
):
    torch.manual_seed(seed)  # reseed here so results don't depend on how much randomness earlier cells consumed
    cfg = BackboneConfig(vocab_size=vocab_size, d_model=d_model, n_layers=n_layers, mixer_kwargs=dict(d_state=d_state))
    model = SequenceBackbone(S4DLayer, cfg)
    tcfg = TrainConfig(
        lr=lr, warmup_steps=max(10, max_steps // 15), max_steps=max_steps,
        eval_every=max(10, max_steps // 10), eval_iters=eval_iters,
        log_every=max(10, max_steps // 5), device=DEVICE,
    )
    trainer = Trainer(model, tcfg, run_name=f"{MODEL_NAME}_{tag}")
    t0 = time.time()
    history = trainer.fit(train_fn, val_fn)
    elapsed = time.time() - t0

    result = {
        "params": model.num_params(),
        "final_val_loss": history["val_loss"][-1],
        "final_val_acc": history["val_acc"][-1],
        "train_seconds": elapsed,
    }
    ALL_RESULTS[tag] = result
    print(f"[{tag}] params={result['params']:,}  val_acc={result['final_val_acc']:.3f}  "
          f"val_loss={result['final_val_loss']:.3f}  ({elapsed:.1f}s)")

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(9, 3))
        axes[0].plot(history["step"], history["val_loss"], marker="o")
        axes[0].set(title=f"{tag}: val loss", xlabel="step")
        axes[1].plot(history["step"], history["val_acc"], marker="o", color="darkorange")
        axes[1].set(title=f"{tag}: val accuracy", xlabel="step", ylim=(0, 1.02))
        fig.tight_layout()
        plt.show()
    return history


## 1. MQAR (associative recall)

The single most predictive synthetic diagnostic in current sub-quadratic
architecture research (Arora et al., *Zoology*, 2024) -- recall a
previously-shown value given its key, repeatedly, at random positions.
S4D's transition is a fixed decay with **no input-dependent selectivity**,
so it has no mechanism to decide "this token is a key I should remember" --
expect this to be the benchmark S4D struggles with most, and the one where
Mamba's selective scan (`02_mamba.ipynb`) should open the largest gap.

In [ ]:
from ssm_lab.benchmarks.mqar import make_mqar_loader

if FAST_DEV_RUN:
    mqar_cfg = dict(batch_size=32, seq_len=32, vocab_size=32, num_kv_pairs=4)
    mqar_train_steps, mqar_d_model, mqar_d_state = 200, 48, 32
else:
    mqar_cfg = dict(batch_size=64, seq_len=256, vocab_size=8192, num_kv_pairs=32)
    mqar_train_steps, mqar_d_model, mqar_d_state = 2000, 128, 64

mqar_train = make_mqar_loader(**mqar_cfg, device=DEVICE, seed=1)
mqar_val = make_mqar_loader(**mqar_cfg, device=DEVICE, seed=2)
_ = run_benchmark("mqar", mqar_cfg["vocab_size"], mqar_train, mqar_val,
                   d_model=mqar_d_model, d_state=mqar_d_state, n_layers=2,
                   max_steps=mqar_train_steps)


## 2. Copying

Reproduces the Jelassi et al. (2024) *"Repeat After Me"* result directly: a
fixed-size state cannot copy an arbitrary string once it's longer than what
that state can hold. We sweep `copy_len` at a **fixed** `d_state` and expect
accuracy to fall off once `copy_len` exceeds roughly what `d_state` complex
numbers can encode -- this is a capacity cliff, not a training-difficulty
curve, and should look qualitatively different from a smooth learning-curve
degradation.

Note on `FAST_DEV_RUN`: with one seed and a few hundred steps, individual
points in this sweep can land in a slower optimization basin than their
neighbors and dent the monotonicity of the curve -- that's optimization
noise from a deliberately truncated dev run, not the capacity effect this
benchmark is actually measuring. The full run (`FAST_DEV_RUN = False`, more
steps, more capacity headroom) is what should show a clean cliff; treat the
dev-run version as "did this crash and are the numbers in a sane range",
not as the result to report.

In [ ]:
from ssm_lab.benchmarks.copying import make_copy_loader

copy_lengths = [2, 4, 8] if FAST_DEV_RUN else [4, 8, 16, 32, 64]
copy_steps = 350 if FAST_DEV_RUN else 1200
copy_d_model, copy_d_state = (32, 16) if FAST_DEV_RUN else (128, 32)  # d_state fixed across the sweep on purpose
copy_vocab = 16 if FAST_DEV_RUN else 64

copy_cliff = []
for copy_len in copy_lengths:
    seq_len = 4 * copy_len  # generous headroom + a little filler after the second copy
    train_fn = make_copy_loader(32, seq_len, copy_vocab, copy_len, device=DEVICE, seed=1)
    val_fn = make_copy_loader(32, seq_len, copy_vocab, copy_len, device=DEVICE, seed=2)
    hist = run_benchmark(f"copy_len{copy_len}", copy_vocab, train_fn, val_fn,
                          d_model=copy_d_model, d_state=copy_d_state, n_layers=2,
                          max_steps=copy_steps, plot=False)
    copy_cliff.append((copy_len, hist["val_acc"][-1]))

fig, ax = plt.subplots(figsize=(6, 3.5))
xs, ys = zip(*copy_cliff)
ax.plot(xs, ys, marker="o")
ax.set(title=f"S4D copying capacity cliff (d_state={copy_d_state} fixed)",
       xlabel="copy_len (tokens to copy)", ylabel="val accuracy", ylim=(0, 1.02))
ax.axvline(copy_d_state, linestyle="--", color="gray", alpha=0.6, label="d_state")
ax.legend()
plt.show()


## 3. Induction heads

Minimal in-context "predict what came after this token, last time" probe
(Olsson et al., 2022). Like MQAR, this needs content-based routing that a
plain decay-only diagonal state doesn't provide -- expected to be a weak
spot for the same structural reason as MQAR, and the literature agrees:
this exact failure mode is why H3 and later Mamba were designed with an
explicit gating/selection mechanism in the first place.

In [ ]:
from ssm_lab.benchmarks.induction_heads import make_induction_loader

if FAST_DEV_RUN:
    ind_cfg = dict(batch_size=32, seq_len=24, vocab_size=16, num_triggers=1)
    ind_steps, ind_d_model, ind_d_state = 200, 32, 16
else:
    ind_cfg = dict(batch_size=64, seq_len=256, vocab_size=64, num_triggers=3)
    ind_steps, ind_d_model, ind_d_state = 1500, 128, 64

ind_train = make_induction_loader(**ind_cfg, device=DEVICE, seed=1)
ind_val = make_induction_loader(**ind_cfg, device=DEVICE, seed=2)
_ = run_benchmark("induction", ind_cfg["vocab_size"], ind_train, ind_val,
                   d_model=ind_d_model, d_state=ind_d_state, n_layers=2, max_steps=ind_steps)


## 4. State tracking: parity and S5 composition

Merrill, Petty & Sabharwal (2024, *"The Illusion of State in State-Space
Models"*): models whose recurrence is a **diagonal, decay-only** linear map
are stuck in TC^0 and provably cannot track state for NC^1-complete problems
like S5 permutation composition -- regardless of how much data or training
they get. S4D's transition is exactly that shape, so an S5-tracking failure
here is not a bug to go fix; it is a predicted, structural result, and the
main reason DeltaNet/DeltaProduct (data-dependent, non-diagonal Householder
updates) and RWKV-7 (a provably more expressive generalized delta rule)
exist as later models in this repo.

In [ ]:
from ssm_lab.benchmarks.state_tracking import (
    S5_VOCAB_SIZE,
    make_parity_loader,
    make_s5_tracking_loader,
)

parity_steps = 200 if FAST_DEV_RUN else 1000
seq_len_parity = 16 if FAST_DEV_RUN else 64
_ = run_benchmark("parity", 2,
                   make_parity_loader(32, seq_len_parity, device=DEVICE, seed=1),
                   make_parity_loader(32, seq_len_parity, device=DEVICE, seed=2),
                   d_model=32 if FAST_DEV_RUN else 96, d_state=16 if FAST_DEV_RUN else 48,
                   n_layers=2, max_steps=parity_steps)

s5_steps = 200 if FAST_DEV_RUN else 1200
seq_len_s5 = 10 if FAST_DEV_RUN else 32
_ = run_benchmark("s5_tracking", S5_VOCAB_SIZE,
                   make_s5_tracking_loader(32, seq_len_s5, device=DEVICE, seed=1),
                   make_s5_tracking_loader(32, seq_len_s5, device=DEVICE, seed=2),
                   d_model=48 if FAST_DEV_RUN else 128, d_state=24 if FAST_DEV_RUN else 64,
                   n_layers=2, max_steps=s5_steps)


## 5. Real language modeling

Everything above is a synthetic algorithmic probe -- deliberately so, to
isolate *why* one architecture beats another. None of it is language, and
"wins at MQAR" isn't by itself evidence this works as a language model.
This section is what actually grounds the comparison in real text, using
the exact same `SequenceBackbone` / `Trainer` as every section above --
only the data source changes.

`ssm_lab.data` prepares a real corpus once (tokenize, train/val split,
write to disk as a `np.memmap`-backed binary) and loads it without ever
reading the whole thing into RAM -- the same code path works whether this
points at the ~1.1MB tiny-shakespeare sample below or a multi-GB
WikiText-103/FineWeb-Edu corpus (see `docs/scaling_up.md`), so which
corpus you use is a config choice, not a rewrite.

In [ ]:
import math

from ssm_lab.data.tokenizer import ByteTokenizer
from ssm_lab.data.text_lm import download_tiny_shakespeare, prepare_corpus, TextLMData

corpus_path = download_tiny_shakespeare("data/tinyshakespeare.txt")
tokenizer = ByteTokenizer()  # vocab=256, zero deps, always round-trips -- see docs/scaling_up.md
with open(corpus_path) as f:
    corpus_text = f.read()

corpus_meta = prepare_corpus(corpus_text, tokenizer, out_dir="data/tinyshakespeare_bin")
print(corpus_meta)
lm_data = TextLMData.from_bin_dir("data/tinyshakespeare_bin")

if FAST_DEV_RUN:
    lm_steps, lm_d_model, lm_d_state, lm_batch, lm_seq_len = 200, 48, 16, 16, 48
else:
    lm_steps, lm_d_model, lm_d_state, lm_batch, lm_seq_len = 3000, 256, 64, 64, 256

lm_train = lm_data.get_batch_fn(lm_batch, lm_seq_len, split="train", device=DEVICE, seed=1)
lm_val = lm_data.get_batch_fn(lm_batch, lm_seq_len, split="val", device=DEVICE, seed=2)
lm_hist = run_benchmark("real_lm_tinyshakespeare", lm_data.vocab_size, lm_train, lm_val,
                         d_model=lm_d_model, d_state=lm_d_state, n_layers=3, max_steps=lm_steps, plot=False)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(lm_hist["step"], lm_hist["val_loss"], marker="o")
axes[0].set(title="real LM: val loss (nats/byte)", xlabel="step")
val_ppl = [math.exp(l) for l in lm_hist["val_loss"]]
axes[1].plot(lm_hist["step"], val_ppl, marker="o", color="darkorange")
axes[1].set(title="real LM: val perplexity", xlabel="step")
axes[1].axhline(lm_data.vocab_size, linestyle="--", color="gray", alpha=0.6, label="chance (uniform)")
axes[1].legend()
fig.tight_layout()
plt.show()

print(f"chance perplexity (uniform over {lm_data.vocab_size} bytes): {lm_data.vocab_size}")
print(f"final val perplexity: {val_ppl[-1]:.2f}")


## 6. Throughput vs. sequence length

The practical payoff of all of the above: a linear-time recurrence should
show close-to-linear wall-clock scaling in sequence length, where a
softmax-attention baseline would show quadratic. This cell measures forward-
pass time only (a real memory-vs-length comparison needs
`torch.cuda.max_memory_allocated()` on an actual GPU -- meaningless on CPU,
so it's skipped automatically when `DEVICE == "cpu"`).

In [ ]:
seq_lengths = [64, 128, 256] if FAST_DEV_RUN else [128, 256, 512, 1024, 2048, 4096]
bench_d_model, bench_n_layers, bench_d_state = (32, 2, 16) if FAST_DEV_RUN else (128, 4, 64)
n_repeats = 3

bench_model = SequenceBackbone(
    S4DLayer,
    BackboneConfig(vocab_size=256, d_model=bench_d_model, n_layers=bench_n_layers, mixer_kwargs=dict(d_state=bench_d_state)),
).to(DEVICE).eval()

timings = []
with torch.no_grad():
    for L in seq_lengths:
        x = torch.randint(0, 256, (8, L), device=DEVICE)
        for _ in range(2):  # warmup
            bench_model(x)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_repeats):
            bench_model(x)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        dt = (time.time() - t0) / n_repeats
        timings.append(dt)
        print(f"seq_len={L:5d}  forward time={dt*1000:8.1f} ms")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(seq_lengths, timings, marker="o")
ax.set(title=f"S4D forward-pass time vs sequence length ({DEVICE})",
       xlabel="sequence length", ylabel="seconds/forward pass")
plt.show()

ALL_RESULTS["throughput"] = {"seq_lengths": seq_lengths, "forward_seconds": timings, "device": DEVICE}


## 7. Scaling infrastructure: checkpointing, accumulation, resume

Everything above ran small enough to finish in one sitting. Training a
model actually worth calling "larger" needs three things this repo's
shared `Trainer`/`SequenceBackbone` support directly (full guide:
`docs/scaling_up.md`):

- **`BackboneConfig(use_gradient_checkpointing=True)`** -- recompute
  activations during backward instead of storing them; verified
  numerically transparent (identical loss/gradients on vs. off) and, in
  this repo's own build, verified to turn an OOM-killed config into a
  successful one.
- **`TrainConfig(grad_accum_steps=k)`** -- k micro-batches accumulated
  into one optimizer step, mathematically equivalent to one k-times-larger
  batch (verified to ~1e-8 numerical agreement).
- **`trainer.save_checkpoint(...)` / `trainer.load_checkpoint(...)`** --
  exact model/optimizer/RNG-state restoration, so training can stop and
  resume across sessions (Kaggle's 12h cap, Colab's variable one) instead
  of needing to fit inside a single one.

Demonstrated together below: train briefly, checkpoint, build a *fresh*
model+trainer (simulating a new session), reload, and confirm the
reloaded model's eval loss is exactly what it was the instant before the
checkpoint was written -- then keep training from there.

In [ ]:
from ssm_lab.models.presets import get_preset

scale_steps = 100 if FAST_DEV_RUN else 2000
preset_name = "tiny" if FAST_DEV_RUN else "small"
scale_cfg, scale_train_overrides = get_preset(preset_name, vocab_size=lm_data.vocab_size)
print(f"preset '{preset_name}': {scale_cfg}, train_overrides={scale_train_overrides}")

scale_train = lm_data.get_batch_fn(16, 48, split="train", device=DEVICE, seed=3)
scale_val = lm_data.get_batch_fn(16, 48, split="val", device=DEVICE, seed=4)

torch.manual_seed(0)
model_session1 = SequenceBackbone(S4DLayer, scale_cfg)
tcfg_session1 = TrainConfig(max_steps=scale_steps, eval_every=scale_steps, eval_iters=5,
                             log_every=scale_steps, device=DEVICE, **scale_train_overrides)
trainer_session1 = Trainer(model_session1, tcfg_session1, run_name="scaling_demo")
trainer_session1.fit(scale_train, scale_val)

loss_before_ckpt, _ = trainer_session1.evaluate(scale_val, n_iters=5)
trainer_session1.save_checkpoint("checkpoints/scaling_demo.pt")
print(f"session 1 done at step {trainer_session1.global_step}, checkpoint saved.")

# a *fixed* batch (not the freshly-advancing scale_val loader) for an exact
# before/after comparison -- two calls to scale_val itself would draw two
# different random batches, which would only be approximately equal and
# would understate how exact this restoration actually is.
fixed_eval_batch = scale_val()
fixed_eval_fn = lambda: fixed_eval_batch  # noqa: E731

loss_before_reload, _ = trainer_session1.evaluate(fixed_eval_fn, n_iters=1)

# --- simulate a fresh session: brand-new model, differently initialized ---
torch.manual_seed(999)
model_session2 = SequenceBackbone(S4DLayer, scale_cfg)
trainer_session2 = Trainer(model_session2, tcfg_session1, run_name="scaling_demo")
trainer_session2.load_checkpoint("checkpoints/scaling_demo.pt")
loss_after_reload, _ = trainer_session2.evaluate(fixed_eval_fn, n_iters=1)

print(f"global_step restored: {trainer_session2.global_step} (expected {trainer_session1.global_step})")
print(f"eval loss on a fixed batch, before save:  {loss_before_reload:.6f}")
print(f"eval loss on that SAME fixed batch, after reload into a differently-initialized fresh model: {loss_after_reload:.6f}")
assert abs(loss_before_reload - loss_after_reload) < 1e-5, "checkpoint restoration should be exact"
print("exact match confirmed.")

trainer_session2.fit(scale_train, scale_val, max_steps=trainer_session2.global_step + max(20, scale_steps // 5))
print(f"resumed training to step {trainer_session2.global_step} without restarting from 0.")


## Summary

Saved to `results/s4d_results.json` -- `20_benchmark_suite.ipynb` reads
every model's results file with this same name pattern to build the
final cross-model comparison table.

In [ ]:
import os

print(f"{'benchmark':<16}{'val_acc':>10}{'val_loss':>12}{'params':>12}")
for tag, r in ALL_RESULTS.items():
    if tag == "throughput":
        continue
    print(f"{tag:<16}{r['final_val_acc']:>10.3f}{r['final_val_loss']:>12.3f}{r['params']:>12,}")

os.makedirs("results", exist_ok=True)
out_path = "results/s4d_results.json" if os.path.isdir("results") else "../results/s4d_results.json"
with open(out_path, "w") as f:
    json.dump(ALL_RESULTS, f, indent=2)
print(f"\nsaved -> {out_path}")


## What to expect (and what it means if you see it)

S4D's transition is a **learned constant** -- no input-dependent gating, no
mechanism to select what to write into or read out of the state. Given that:

- **Copying**: should hold up until `copy_len` approaches `d_state`, then
  fall off a cliff -- a *capacity* limit, cleanly visible in the sweep above.
- **MQAR / induction heads**: expected to be weak. Both need "notice this
  specific token, remember it, retrieve it on demand" -- content-based
  routing a fixed-decay recurrence has no mechanism for.
- **Parity / S5 tracking**: expected to be weak-to-failing, especially S5.
  This isn't a training bug to chase -- it's the Merrill et al. result,
  reproduced.
- **Real language modeling**: perplexity should drop well below chance
  (`vocab_size`) even at FAST_DEV_RUN scale -- this repo's own build took a
  69K-param S4D from chance (256) to ~11.4 val perplexity in 200 steps on
  real tiny-shakespeare. This is the section that actually answers "does
  this work as a language model," as distinct from every synthetic
  benchmark above answering "why does/doesn't this architecture do X."

If your numbers land roughly here, S4D is working *correctly* and the
notebook has done its job: it's given the next 13 models something concrete
to beat, on benchmarks specifically chosen because they separate
architectures for *structural* reasons rather than by luck of
hyperparameters. `02_mamba.ipynb` (selective, input-dependent scan) is the
first model expected to move the MQAR/induction numbers meaningfully.

Section 7 demonstrated, not just described, that this same backbone/trainer
scales past benchmark size: gradient checkpointing, gradient accumulation,
and exact checkpoint/resume are all verified (here and in
`tests/test_grad_checkpointing.py`, `test_grad_accum.py`,
`test_checkpoint.py`) rather than assumed. `docs/scaling_up.md` is the full
guide, including named scaling presets with real (not estimated) parameter
counts computed from this exact codebase.